# 04 Subset metacells

Second-pass metacell run restricted to one lineage. Input is the **global** `clean_cells.h5ad`
plus the block annotations written by the global step `03`.

Differences from the global run:

1. **No per-cell QC.** MAD / miQC / Scrublet / excluded-fraction filtering already happened
   upstream. Cells arriving here are clean by construction; re-filtering on a subset would
   recompute MAD thresholds against a different population and silently drop real biology.
2. **Subset by `broad_label`** from `global_metacell_block_annotations.csv`.
3. **Stale MC2 fields are stripped** before rerunning. The global run left `metacell`,
   `metacell_name`, `lateral_gene`, `selected_gene`, `rare_gene_module`, etc. on the object;
   MC2 will happily reuse those masks if they are still present.
4. **Smaller target metacell size** so the subset is resolved more finely than the global run.
5. **Metacell names are prefixed** (`T_M...`) so subset and global IDs can never collide.

Lateral/noisy lists are carried over unchanged. The TCR V/J segments matter *more* here, not
less: within a single-lineage object, clonal TCR signal is exactly what would hijack the graph.


In [ ]:
#import libraries
import scanpy as sc
import anndata as ad
import metacells as mc
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sb
import os
from scipy.stats import median_abs_deviation


### Target metacell size and UMIs for a subset

Graining level gamma = cells / metacells, set directly by `TARGET_METACELL_SIZE`.
The benchmarked range is gamma 10 to 50 (Bilous et al., BMC Bioinformatics 2022;23:336);
50 is a ceiling rather than a recommendation.

Gamma follows cell count, not cell type. What changes between lineages is depth,
so `TARGET_METACELL_UMIS` is what gets re-derived per subset:

| Subset | Governing consideration | Suggested SIZE | UMIS |
| --- | --- | --- | --- |
| T cells | Largest subset, gamma can sit high | 40-50 | SIZE x median |
| Myeloid | Deeper libraries, fewer cells | 30-40 | SIZE x median |
| B cells | Usually smallest of the three | 24-32 | SIZE x median |

The chunk further down prints the median post-exclusion UMIs for this subset and
the implied UMI target at three sizes. Set it from there, not from the global run.
Full reasoning: docs/06_parameters.md.

In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
# Every path and tunable comes from config/config.yaml, so this notebook holds
# no sample ids and no absolute paths.
import os, yaml

CFG_DIR = os.path.abspath(os.path.join("..", "config"))
CFG  = yaml.safe_load(open(os.path.join(CFG_DIR, "config.yaml")))
ROOT = os.path.abspath("..")
rp = lambda p: p if os.path.isabs(p) else os.path.join(ROOT, p)

PROJECT  = CFG["project"]["name"]
h5ad_dir = rp(os.path.join(CFG["paths"]["h5ad"], "subset", "metacell"))
qc_dir   = rp(os.path.join(CFG["paths"]["csvs"], "global", "qc", "metacell"))
os.makedirs(h5ad_dir, exist_ok=True); os.makedirs(qc_dir, exist_ok=True)

M = CFG["metacells_subset"]
# target cells per metacell. Sets the graining level gamma = cells / metacells.
TARGET_METACELL_SIZE     = M["target_metacell_size"]
# target UMIs per metacell. Whichever of the two binds first is the one that acts.
TARGET_METACELL_UMIS     = M["target_metacell_umis"]
# metacells per pile: how the divide-and-conquer step chunks the data
TARGET_METACELLS_IN_PILE = M["target_metacells_in_pile"]
MIN_PILE                 = M["min_pile"]
MAX_PILE                 = M["max_pile"]
RANDOM_SEED              = M["random_seed"]

# which lineage to pull out of the global annotation, and how strictly
SUBSET_LABELS = ["Lineage A"]        # broad_label value(s) to keep
MC_PREFIX     = CFG["metacells_subset"]["mc_prefix"]
T_MIN         = CFG["metacells_subset"]["purity_min"]
OTHER_MAX     = CFG["metacells_subset"]["purity_other_max"]
MARGIN_THRESH = CFG["metacells_subset"]["margin_thresh"]

# the global run's outputs, which this notebook reads
global_h5ad = rp(os.path.join(CFG["paths"]["h5ad"], "global", "metacell"))
global_csv  = rp(os.path.join(CFG["paths"]["csvs"], "global", "metacell"))

QC_XLSX_PATH = os.path.join(qc_dir, f"{PROJECT}_subset_metacell_QC.xlsx")

# MC2 parallelises over piles; take the scheduler's allocation when there is one
n_processors = int(os.environ.get("LSB_DJOB_NUMPROC", os.cpu_count() or 4))
mc.ut.set_processors_count(n_processors)
print(f"Using {n_processors} processors")


def write_xlsx_sheets(path, sheets):
    """Create-or-append sheets to an .xlsx workbook. Idempotent: a sheet of the
    same name is replaced, so a rerun of one chunk does not duplicate tabs."""
    mode = "a" if os.path.exists(path) else "w"
    kw = {"if_sheet_exists": "replace"} if mode == "a" else {}
    with pd.ExcelWriter(path, engine="openpyxl", mode=mode, **kw) as xw:
        for name, df in sheets.items():
            df.to_excel(xw, sheet_name=name[:31], index=False)
    print(f"Wrote {list(sheets)} -> {path}")


## 1. Load global cell level obj with annotations mapped


In [ ]:
cdata_path = os.path.join(global_h5ad, f"{PROJECT}.clean_cells.h5ad")
global_cdata = ad.read_h5ad(cdata_path)
mc.ut.set_name(global_cdata, f"{PROJECT}.cells")
print("Shape:", global_cdata.shape, "| cells:", global_cdata.n_obs, "| genes:", global_cdata.n_vars)
print(global_cdata.obs["sample_origin"].value_counts())


## 1b. Annotation confidence check before subsetting

Every cell inherits its metacell's block label, and every metacell inherits its block's lineage
call. A block called on a thin margin therefore drags all of its cells into the wrong subset at
once, and nothing downstream will flag it: the subset object looks clean, the metacells rebuild
fine, and the DEGs come out of a contaminated population.

`score_margin` is the gap between the best and second-best lineage z-score for a block, computed
in the global 03. `best_score` is the top z itself. Two failure modes are worth separating:

- **low margin, high best_score** - two lineages both fit. Usually a real doublet-ish or
  transitional block, or shared markers (NK vs cytotoxic T, moDC vs Mo/Mac).
- **low best_score** - nothing fits. Already labelled `Undefined` upstream, so these do not
  silently enter the subset, but they are worth looking at because a genuine T block can land
  here if its markers were mostly lateral.

The marker table below is the arbiter. Read it, then fill in the override dict in 1c.


In [ ]:
annot_path = os.path.join(global_csv, "global_metacell_block_annotations.csv")
mc_annot = pd.read_csv(annot_path, dtype=str)
print("metacells annotated globally:", len(mc_annot))

# block-level scores written by the global 05
blockmap_xlsx = os.path.join(global_csv, "global_module_block_map.xlsx")
block_qc = pd.read_excel(blockmap_xlsx, sheet_name="block_celltype_mapping", dtype={"block": str})

# workbooks written before the sign fix store margin as (second - best), i.e. negative.
# abs() makes this chunk correct against either version: positive = confident, either way.
block_qc["score_margin"] = block_qc["score_margin"].abs()

n_mc   = mc_annot.groupby("mc_block").size().rename("n_metacells")
mc2cell = global_cdata.obs["metacell_name"].astype(str).map(
    dict(zip(mc_annot["metacell_name"], mc_annot["mc_block"]))
)
n_cell = mc2cell.value_counts().rename("n_cells")

block_qc = (block_qc.set_index("block").join(n_mc).join(n_cell).reset_index())
block_qc["flag"] = np.where(
    block_qc["assigned_celltype"] == "Undefined", "undefined",
    np.where(block_qc["score_margin"] < MARGIN_THRESH, "low_margin", ""))

print(f"\nblocks within {MARGIN_THRESH} z of their second-best call: "
      f"{int((block_qc['flag'] == 'low_margin').sum())} of {len(block_qc)}")
print(block_qc.sort_values("score_margin")
      [["block","assigned_celltype","best_score","second_score","score_margin",
        "n_metacells","n_cells","top_module","flag"]].to_string(index=False))

# how many cells the ambiguous blocks actually account for. If this is a fraction of a percent,
# the whole question is academic and the default subset is fine as-is.
amb = block_qc.loc[block_qc["flag"] != "", "n_cells"].sum()
print(f"\ncells in flagged blocks: {amb:,.0f} "
      f"({100*amb/global_cdata.n_obs:.2f}% of clean cells)")


### Marker evidence for the flagged blocks

Mean CP10K log1p per block for a compact diagnostic panel, so the call can be made on expression
rather than on the z-score that was already ambiguous. For a subset the decisive comparison is
`CD3D`/`CD3E`/`TRAC` against `KLRF1`/`NCAM1` (NK), `LYZ`/`CD14` (Mo/Mac) and `MS4A1` (B) - a real
T block should be unambiguous on CD3 even when its state markers are mixed.


In [ ]:
#Review top 5 modules from block mapping files apart from this to confirm whether block annotations are accurate

diag_markers = {
    "T":        ["CD3D","CD3E","CD3G","TRAC","CD2"],
    "NK":       ["KLRF1","NCAM1","NKG7","GNLY","KLRD1"],
    "B":        ["MS4A1","CD79A","BANK1"],
    "Plasma":   ["MZB1","JCHAIN"],
    "Mo/Mac":   ["CD14","LYZ","FCN1","S100A9"],
    "DC":       ["CD1C","FCER1A","LILRA4"],
    "Platelet": ["PPBP","PF4"],
    "Cycling":  ["MKI67","STMN1"],
}
present = {k: [g for g in v if g in global_cdata.var_names] for k, v in diag_markers.items()}
print({k: len(v) for k, v in present.items()})

flagged = block_qc.loc[block_qc["flag"] != "", "block"].astype(str).tolist()
if flagged:
    genes = sorted({g for v in present.values() for g in v})
    sub   = global_cdata[:, genes]
    lib   = np.asarray(global_cdata.X.sum(axis=1)).flatten()
    Xn    = np.log1p(np.asarray(sub.X.todense()) / np.maximum(lib, 1)[:, None] * 1e4)
    expr  = pd.DataFrame(Xn, columns=genes)
    expr["block"] = mc2cell.to_numpy()

    per_block = expr.dropna(subset=["block"]).groupby("block").mean(numeric_only=True)
    panel_mean = pd.DataFrame(
        {k: per_block[v].mean(axis=1) for k, v in present.items() if v}
    )
    show = panel_mean.loc[[b for b in flagged if b in panel_mean.index]]
    show = show.join(block_qc.set_index("block")[["assigned_celltype","score_margin","n_cells"]])
    print("\nmean CP10K log1p per lineage panel, flagged blocks only:")
    print(show.round(2).to_string())
else:
    print("no flagged blocks, nothing to review")


## 1c. Reassign mislabelled blocks

Fill `BLOCK_RELABEL` from the table above, then rerun. Keys are block IDs as strings, values are
the new `broad_label`. `MC_RELABEL` does the same for an individual metacell when only part of a
block is wrong, and is applied after the block-level pass so it wins.

Both dicts default to empty, so leaving them alone reproduces the unmodified global annotation
exactly. The overrides are written to the QC workbook because a hand-curated label is a methods
decision, not a preprocessing detail ; it needs to be reportable.

Two things not to do here. Do not relabel a block to move borderline cells *into* the subset on
the theory that more cells is better: a contaminated block hurts the DEGs more than a slightly
undersized subset does. And do not relabel `Undefined` blocks wholesale - if nothing scored above
zero, the honest read is that the block has no confident identity, and forcing one in is the sort
of choice that gets caught at review.


Relabelling here means the workbooks and the annotated RDS from `03` are stale.
Rerun the last three chunks of `03` before continuing.


In [ ]:
# BLOCK_RELABEL = {
#     "7": "T cells",       # M28 (CD28, ICOS, CAMK4) + M23 (ITK, THEMIS, PRKCQ, BCL11B) naive/CM CD4; scores ~0 from CD3 dropout
#     "5": "Granulocytes",  # M5 neutrophil (CD177, ALPL, CYP4F3, ADGRG3); no neutrophil genes in the mast-only signature
#     "1": "MK/Ery",        # M24 MK (P2RY12, RGS18, NFE2, MEIS1) + M32/M27 ery (HEMGN, CA2, RHAG); called on GATA2 alone
# }
# MC_RELABEL = {
#     # "M1885.31": "T cells",
# }

# mc_annot["broad_label_original"] = mc_annot["broad_label"]
# mc_annot["relabel_source"] = ""

# if BLOCK_RELABEL:
#     hit = mc_annot["mc_block"].isin(BLOCK_RELABEL)
#     mc_annot.loc[hit, "broad_label"]    = mc_annot.loc[hit, "mc_block"].map(BLOCK_RELABEL)
#     mc_annot.loc[hit, "relabel_source"] = "block"
# if MC_RELABEL:
#     hit = mc_annot["metacell_name"].isin(MC_RELABEL)
#     mc_annot.loc[hit, "broad_label"]    = mc_annot.loc[hit, "metacell_name"].map(MC_RELABEL)
#     mc_annot.loc[hit, "relabel_source"] = "metacell"

# changed = mc_annot[mc_annot["broad_label"] != mc_annot["broad_label_original"]]
# print(f"metacells relabelled: {len(changed)}")
# if len(changed):
#     print(changed.groupby(["broad_label_original","broad_label","relabel_source"])
#           .size().rename("n_metacells").reset_index().to_string(index=False))

# # guard against a typo'd block ID silently doing nothing
# _unknown = set(BLOCK_RELABEL) - set(block_qc["block"].astype(str))
# assert not _unknown, f"block IDs not in block_qc: {sorted(_unknown)}"

# relabel_log = pd.DataFrame(
#     [{"level":"block","id":k,"new_label":v} for k,v in BLOCK_RELABEL.items()] +
#     [{"level":"metacell","id":k,"new_label":v} for k,v in MC_RELABEL.items()]
# ) if (BLOCK_RELABEL or MC_RELABEL) else pd.DataFrame(columns=["level","id","new_label"])
# write_xlsx_sheets(QC_XLSX_PATH, {"block_qc": block_qc, "manual_relabels": relabel_log})


# Subset for cell type of interest

`broad_label` is joined on `metacell_name` using the (possibly corrected) `mc_annot` table from
1c, so global outlier cells (`metacell < 0`, no metacell name) fall out here by construction.
Metacells present in `clean_cells.h5ad` but absent from the module heatmap come back as
`Unassigned`; that count should be 0, and anything else means metacells were dropped somewhere
between `mdata` and `thmz` in the global 03.


In [ ]:
# --- purity thresholds -------------------------------------------------------
# a metacell enters the subset if it carries T signal and carries no foreign program.
# absolute ceilings, not a T-minus-other margin: the margin punishes shallow metacells
# (T=0.40 / Mono=0.07 is clean but has a gap of 0.33) and that is a depth artefact, not identity.

lut = dict(zip(mc_annot["metacell_name"], mc_annot["broad_label"]))
mc_lab = lut
mc_name_global = global_cdata.obs["metacell_name"].astype(str)
in_metacell    = global_cdata.obs["metacell"].to_numpy().astype(int) >= 0

lab = mc_name_global.map(lut).fillna("Unassigned").to_numpy().astype(object)
lab[~in_metacell] = "Outlier"
global_cdata.obs["global_broad_label"] = pd.Categorical(lab)
print(global_cdata.obs["global_broad_label"].value_counts(dropna=False))

global_cdata.obs["global_mc_block"] = pd.Categorical(
    mc_name_global.map(dict(zip(mc_annot["metacell_name"], mc_annot["mc_block"]))).fillna("NA"))
_margin = dict(zip(block_qc["block"].astype(str), block_qc["score_margin"]))
global_cdata.obs["global_anno_margin"] = (
    global_cdata.obs["global_mc_block"].astype(str).map(_margin).astype(float))

# --- metacell purity gate ----------------------------------------------------
# block labels apply to every metacell in the block, and the largest T blocks hold 664 and 659.
# a myeloid / B / DC metacell inside one inherits "T cells" and enters the subset unchallenged,
# which is what put monocyte, B, cDC and platelet modules in the subset module run.
PURITY = {
    "T":    ["CD3D","CD3E","CD3G","CD2","TRAC"],
    "Mono": ["LYZ","S100A8","S100A9","FCN1","CD14","CST3","AIF1","LST1"],
    "B":    ["MS4A1","CD79A","BANK1","IGKC"],
    "DC":   ["CD1C","FCER1A","CLEC4C","LILRA4","IRF8"],
    "Plt":  ["PPBP","PF4","PF4V1","TREML1"],
    "HSPC": ["CD34","PROM1"],
}
# reported only, never gated on. KLRD1 is CD94 and sits on CD8 Tem/Temra and gdT; gating on it
# rejected block 17. the T/NK call is made at block level by the CD3 veto in 03.
NK_PANEL = ["KLRF1","NCAM1","SPON2"]

pg    = {k: [g for g in v if g in global_cdata.var_names] for k, v in PURITY.items()}
nkp   = [g for g in NK_PANEL if g in global_cdata.var_names]
genes = sorted({g for v in pg.values() for g in v} | set(nkp))
print("purity panel genes found:", {k: len(v) for k, v in pg.items()}, "| NK diag:", len(nkp))

lib = np.asarray(global_cdata.X.sum(axis=1)).flatten()
Xn  = np.log1p(np.asarray(global_cdata[:, genes].X.todense())
               / np.maximum(lib, 1)[:, None] * 1e4)

expr = pd.DataFrame(Xn, columns=genes, index=global_cdata.obs_names)
expr["mc"] = np.where(in_metacell, mc_name_global.to_numpy(), None)
per_mc     = expr.dropna(subset=["mc"]).groupby("mc").mean(numeric_only=True)
panel_mc   = pd.DataFrame({k: per_mc[v].mean(axis=1) for k, v in pg.items() if v})

other   = panel_mc.drop(columns=["T"]).max(axis=1)
mc_pure = (panel_mc["T"] >= T_MIN) & (other <= OTHER_MAX)
panel_mc["NK_diag"] = per_mc[nkp].mean(axis=1) if nkp else np.nan

_lab = pd.Series(panel_mc.index.map(mc_lab), index=panel_mc.index)
_isT = _lab.isin(SUBSET_LABELS)

print("\nT score, all metacells:")
print(panel_mc["T"].describe().round(2).to_string())
print("\nworst contaminant panel within T-labelled blocks (pick OTHER_MAX off the knee):")
print(other[_isT].describe(percentiles=[.5,.75,.9,.95,.98,.99]).round(2).to_string())
print(f"\nmetacells failing purity: {int((~mc_pure).sum()):,} of {len(mc_pure):,}")

# concentrated failures mean a block label is still wrong; diffuse means stray metacells,
# which is what the gate exists for
_pure_by_cell = mc_name_global.map(mc_pure)
fail_blk = (global_cdata.obs.loc[in_metacell]
            .assign(pure=_pure_by_cell[in_metacell].to_numpy())
            .query("global_broad_label in @SUBSET_LABELS")
            .groupby("global_mc_block", observed=True)["pure"]
            .agg(n_cells="size", n_fail=lambda s: int((~s.astype(bool)).sum())))
fail_blk["pct_fail"] = (100 * fail_blk["n_fail"] / fail_blk["n_cells"]).round(1)
print("\nper-block purity failure, T-labelled blocks:")
print(fail_blk.sort_values("pct_fail", ascending=False).to_string())

# dropped metacells should look foreign, not merely dim. high Mono/B here = gate working;
# low everything with low T = the floor is too high and real cells are being lost.
print("\nmean panel of dropped metacells inside T-labelled blocks:")
print(panel_mc.loc[(~mc_pure) & _isT].mean().round(2).to_string())

# --- subset ------------------------------------------------------------------
keep = global_cdata.obs["global_broad_label"].isin(SUBSET_LABELS).to_numpy()
n_label = int(keep.sum())
keep = keep & _pure_by_cell.astype("boolean").fillna(False).to_numpy(dtype=bool)
print(f"\nlabel only: {n_label:,} cells -> after purity gate: {int(keep.sum()):,} "
      f"({100*(n_label-keep.sum())/max(n_label,1):.2f}% dropped)")

if "RARE_CELLS_TO_DROP" in dir() and RARE_CELLS_TO_DROP:
    keep = keep & ~global_cdata.obs_names.isin(RARE_CELLS_TO_DROP)
    print(f"dropped {len(RARE_CELLS_TO_DROP)} rare-module cells")

subset_data_cdata = global_cdata[keep].copy()
print(f"\n{SUBSET_LABELS} -> {subset_data_cdata.n_obs} cells "
      f"({100*subset_data_cdata.n_obs/global_cdata.n_obs:.1f}% of clean cells)")

low = (subset_data_cdata.obs["global_anno_margin"] < MARGIN_THRESH).sum()
print(f"cells from blocks within {MARGIN_THRESH} z of a second call: {low:,} "
      f"({100*low/subset_data_cdata.n_obs:.2f}% of subset)")
print(subset_data_cdata.obs["sample_origin"].value_counts())

write_xlsx_sheets(QC_XLSX_PATH, {"purity_gate": fail_blk.reset_index()})


In [ ]:
mc_block = dict(zip(mc_annot["metacell_name"], mc_annot["mc_block"]))
mc_lab   = dict(zip(mc_annot["metacell_name"], mc_annot["broad_label"]))

pm = panel_mc.copy()
pm["block"]  = pm.index.map(mc_block)
pm["label"]  = pm.index.map(mc_lab)
pm["worst"]  = panel_mc.drop(columns=["T"]).idxmax(axis=1)
pm["fail_floor"]  = pm["T"] < T_MIN
pm["fail_margin"] = (pm["T"] - panel_mc.drop(columns=["T"]).max(axis=1)) < T_MARGIN

tp = pm[pm["label"].isin(SUBSET_LABELS)]
print(tp.groupby("block")[["fail_floor","fail_margin"]].mean().round(2).to_string())

for b in ["9","11","17"]:
    d = tp[tp["block"] == b]
    print(f"\nblock {b}, n_mc {len(d)}")
    print(d[list(panel_mc.columns)].mean().round(2).to_string())
    print(d.loc[d.fail_floor | d.fail_margin, "worst"].value_counts().to_string())


In [ ]:
#Check UMI distribution within subset to determine if TARGET_METACELL_UMIS/SIZE needs to be updated at top of the sheet; rough estimates

#if TARGET_METACELL_UMIS is far above TARGET_METACELL_SIZE x median UMIs the size target binds and metacells stay ~36 cells; if it is
# far below, the UMI cap binds and metacells come out much smaller than 36 cells
umis = np.asarray(subset_data_cdata.X.sum(axis=1)).flatten()
print(f"n_cells={len(umis):,}  median={np.median(umis):,.0f}  "
      f"IQR={np.percentile(umis,25):,.0f}-{np.percentile(umis,75):,.0f}")
for s in [24, 36, 48]:
    print(f"  size={s} -> umis={round(s*np.median(umis)/1000)*1000:,}")


## 1d. Strip stale MC2 fields from the global run

MC2 does not clear its own annotations. If `excluded_gene` / `lateral_gene` / `selected_gene` /
`metacell` survive into this object, `exclude_genes` and the DAC pipeline will reuse the global
masks instead of recomputing them on the subset. Anything worth keeping is renamed with a
`global_` prefix first.


In [ ]:
keep_as_global = ["metacell", "metacell_name", "metacell_level", "dissolved",
                  "most_similar", "most_similar_name"]
for c in keep_as_global:
    if c in subset_data_cdata.obs:
        subset_data_cdata.obs[f"global_{c}"] = subset_data_cdata.obs[c].values

stale_obs = keep_as_global + ["excluded_cell", "clean_cell", "rare_cell", "rare_gene_module",
                              "cells_rare_gene_module", "pile", "candidate",
                              "excluded_gene_fraction", "properly_sampled_cell",
                              "mad_outlier", "miqc_compromised"]
stale_var = ["excluded_gene", "clean_gene", "lateral_gene", "noisy_gene", "selected_gene",
             "rare_gene", "rare_gene_module", "full_gene_index", "top_feature_gene",
             "marker_gene", "significant_inner_folds_count", "selected_gene_umis",
             "pre_selected_gene", "strong_gene_module"]
stale_var += [c for c in subset_data_cdata.var.columns if c.startswith("rare_gene_module_")]

dropped_obs = [c for c in stale_obs if c in subset_data_cdata.obs]
dropped_var = [c for c in stale_var if c in subset_data_cdata.var]
subset_data_cdata.obs.drop(columns=dropped_obs, inplace=True)
subset_data_cdata.var.drop(columns=dropped_var, inplace=True)
# global_broad_label / global_mc_block / global_anno_margin are deliberately kept
for store in (subset_data_cdata.obsp, subset_data_cdata.varp, subset_data_cdata.uns):
    for k in list(store.keys()):
        if k not in ("__name__",):
            del store[k]

print("dropped obs:", dropped_obs)
print("dropped var:", dropped_var)
print("remaining obs:", list(subset_data_cdata.obs.columns))
print("remaining var:", list(subset_data_cdata.var.columns))

mc.ut.set_name(subset_data_cdata, f"{PROJECT}.subset.cells")


## 2. Per-cell QC metrics

`total_umis`, `n_genes`, distribution for subset level.

`percent_mito` is **not** recomputed here: the global `exclude_genes` already removed every
`MT-*` gene, so a fresh `str.startswith("MT-")` mask matches nothing and would silently return
0% for every cell. The global value is carried over as `global_percent_mito` instead.

`total_umis` on this object is the post-exclusion total (MT, MALAT1, HB genes already gone), so
it is lower than the raw library size. That is the number the metacell UMI target is compared
against, which is why the suggested `TARGET_METACELL_UMIS` below is computed from it.


In [ ]:
total_umis = np.asarray(subset_data_cdata.X.sum(axis=1)).flatten()
n_genes    = np.asarray((subset_data_cdata.X > 0).sum(axis=1)).flatten()

subset_data_cdata.obs["total_umis"] = total_umis
subset_data_cdata.obs["n_genes"]    = n_genes

mito_mask = subset_data_cdata.var_names.str.startswith("MT-")
print("MT- genes still present (expect 0):", int(mito_mask.sum()))
if "percent_mito" in subset_data_cdata.obs:
    subset_data_cdata.obs["global_percent_mito"] = subset_data_cdata.obs["percent_mito"].values

for label, v in [("total_umis", total_umis), ("n_genes", n_genes)]:
    qs = np.percentile(v, [1, 5, 25, 50, 75, 95, 99])
    print(label, "P1/5/25/50/75/95/99:", np.round(qs, 2))


## 5. Excluded / lateral / noisy gene lists

Lists are byte-identical to the global run so the two levels stay comparable.
Most of `Excluded_gnames` is already gone (dropped by the global
`extract_clean_data`), so the count printed below is much smaller than in `02`.
That is expected.

The zero-count gene filter added here matters more than it did globally:
subsetting to one lineage leaves several thousand genes at exactly 0 UMIs, and
those destabilise the variance-mean LOESS fit in `05`.

TCR/BCR V/J segments are the list to check first on a lineage subset. See
docs/05_gene_lists.md.

In [ ]:
Excluded_gnames = [
    "MALAT1", "NEAT1",
    "HBB", "HBA1", "HBA2", "HBD", "HBM",
    "MT-ND1","MT-ND2","MT-CO1","MT-CO2","MT-ATP8","MT-ATP6","MT-CO3","MT-ND3","MT-ND4L",
    "MT-ND4","MT-ND5","MT-ND6","MT-CYB",
    "MTRNR2L11","MTRNR2L12","MTRNR2L13","MTRNR2L6","MTRNR2L10","MTRNR2L8","MTRNR2L7",
    "MTRNR2L5","MTRNR2L4","MTRNR2L1","MTRNR2L3",
]

#dropped the old 'MT1.*' pattern, it matched metallothioneins (MT1A/MT2A) - which are stress-response genes
Excluded_gpatterns = [
    "^MT-.*",
    "^MTRNR.*",
]

#Cell cycle (Tirosh 2016 Science 352:189 = Seurat cc.genes/regev_lab list)
CC_S = ["MCM5","PCNA","TYMS","FEN1","MCM2","MCM4","RRM1","UNG","GINS2","MCM6","CDCA7","DTL",
        "PRIM1","UHRF1","MLF1IP","HELLS","RFC2","RPA2","NASP","RAD51AP1","GMNN","WDR76","SLBP",
        "CCNE2","UBR7","POLD3","MSH2","ATAD2","RAD51","RRM2","CDC45","CDC6","EXO1","TIPIN",
        "DSCC1","BLM","CASP8AP2","USP1","CLSPN","POLA1","CHAF1B","BRIP1","E2F8"]
CC_G2M = ["HMGB2","CDK1","NUSAP1","UBE2C","BIRC5","TPX2","TOP2A","NDC80","CKS2","NUF2","CKS1B",
          "MKI67","TMPO","CENPF","TACC3","FAM64A","SMC4","CCNB2","CKAP2L","CKAP2","AURKB","BUB1",
          "KIF11","ANP32E","TUBB4B","GTSE1","KIF20B","HJURP","CDCA3","HN1","CDC20","TTK","CDC25C",
          "KIF2C","RANGAP1","NCAPD2","DLGAP5","CDCA2","CDCA8","ECT2","KIF23","HMMR","AURKA",
          "PSRC1","ANLN","LBR","CKAP5","CENPE","CTCF","NEK2","G2E3","GAS2L3","CBX5","CENPA"]
# Stress/immediate-early/dissociation (van den Brink 2017 Nat Methods 14:935; O'Flanagan 2019 Genome Biol 20:210)
STRESS = ["FOS","FOSB","JUN","JUNB","JUND","EGR1","IER2","ATF3","ZFP36","SOCS3","NR4A1","DUSP1",
          "PPP1R15A","HSPA1A","HSPA1B","HSPA6","HSPB1","HSP90AA1","HSP90AB1","DNAJB1","DNAJA1","BAG3","UBC"]
#Sex-linked
SEX = ["XIST","RPS4Y1","DDX3Y","UTY","KDM5D","EIF1AY","NLGN4Y","USP9Y","ZFY"]
#Platelet/megakaryocyte ambient
PLATELET = ["PPBP","PF4","GP9","GP1BA","ITGA2B","TUBB1","NRGN","CAVIN2","GNG11"]

Lateral_gnames = (CC_S + CC_G2M + STRESS + SEX + PLATELET +
    ["HLA-F","HLA-G","HLA-A","HLA-E","HLA-C","HLA-B","HLA-DRB5","HLA-DRB1","HLA-DQA1","HLA-DQB1",
     "HLA-DQB1-AS1","HLA-DQA2","HLA-DQB2","HLA-DOB","HLA-DMB","HLA-DMA","HLA-DOA","HLA-DPA1",
     "HLA-DPB1","JCHAIN"])
Lateral_gpatterns = [
    # (Brief Funct Genomics 2022 22:263; scRepertoire quietTCRgenes), TRAC/TRBC kept
    "^TRAV.*","^TRBV.*","^TRGV.*","^TRDV.*","^TRAJ.*","^TRBJ.*","^TRDJ.*","^TRBD.*","^TRDD.*",
    #BCR V/J segments and immunoglobulin
    "^IGKV.*","^IGKJ.*","^IGHV.*","^IGHJ.*","^IGLJ.*","^IGLV.*",
    #ribosomal (Baran 2019)
    "^RPS.*","^RPL.*","^RPP.*",
    #heat-shock family
    "^HSPA.*","^HSPB.*",
]

Noisy_gnames = STRESS + ["JCHAIN","HLA-A","HLA-B","HLA-C","HLA-E","HLA-DRB1","HLA-DQA1","HLA-DQB1"]
Noisy_gpatterns = ["^IGHM.*","^IGHA.*","^IGHG.*","^IGKV.*","^IGKJ.*","^IGHV.*","^IGHJ.*","^IGLJ.*","^IGLV.*","^HSPA.*","^HSPB.*"]

mc.pl.exclude_genes(
    adata=subset_data_cdata,
    excluded_gene_names=Excluded_gnames,
    excluded_gene_patterns=Excluded_gpatterns,
    random_seed=RANDOM_SEED,
)

# genes with zero UMIs anywhere in this lineage carry no information and destabilise the
# variance/mean loess in 05
gene_umis = np.asarray(subset_data_cdata.X.sum(axis=0)).flatten()
zero_gene = gene_umis == 0
subset_data_cdata.var["excluded_gene"] = (
    subset_data_cdata.var["excluded_gene"].to_numpy().astype(bool) | zero_gene
)
print("Excluded genes:", int(subset_data_cdata.var["excluded_gene"].sum()),
      f"(of which {int(zero_gene.sum())} are zero-count in this subset)")


## 6. Extract clean data

No cell-level filter is applied: every cell here already survived the global MAD / miQC /
Scrublet / excluded-fraction pipeline. `excluded_cell` is set explicitly to all-False so
`extract_clean_data` has the field it expects and the gene filter is the only thing acting.


In [ ]:
subset_data_cdata.obs["excluded_cell"] = np.zeros(subset_data_cdata.n_obs, dtype=bool)

# subset composition carried through to the QC workbook, replaces the global attrition table
attrition = (
    subset_data_cdata.obs.groupby("sample_origin", observed=True)
    .size().rename("n_subset_cells").reset_index()
)
attrition["pct_of_subset"] = (100 * attrition["n_subset_cells"] / subset_data_cdata.n_obs).round(2)
print(attrition.to_string(index=False))

clean = mc.pl.extract_clean_data(adata=subset_data_cdata)
mc.ut.set_name(clean, f"{PROJECT}.subset.clean")
print(f"\nCells: {subset_data_cdata.n_obs} -> {clean.n_obs} ({clean.n_obs/subset_data_cdata.n_obs:.2%})")
print(f"Genes: {subset_data_cdata.n_vars} -> {clean.n_vars} ({clean.n_vars/subset_data_cdata.n_vars:.2%})")


## 7. Mark lateral & noisy genes on the clean object


In [ ]:
mc.pl.mark_lateral_genes(
    adata=clean, lateral_gene_names=Lateral_gnames, lateral_gene_patterns=Lateral_gpatterns,
)
mc.pl.mark_noisy_genes(
    adata=clean, noisy_gene_names=Noisy_gnames, noisy_gene_patterns=Noisy_gpatterns,
)
print("lateral:", int(clean.var["lateral_gene"].sum()), "| noisy:", int(clean.var["noisy_gene"].sum()))


## 8. Select feature genes + QC that no lateral gene leaked in

The metacells docs recommend checking the `selected_gene` mask against `lateral_gene`: any lateral
gene that got selected would degrade the metacells and should be added to the lateral list and the
run repeated.

Expect substantially fewer selected genes than the global run. Between-lineage variance is gone,
so what is left is within-lineage structure (naive/memory/effector, CD4/CD8, cytotoxicity,
exhaustion, interferon). If the count drops below ~200 it would materially degrade the metacells, and the cause is upstream


In [ ]:
selected = mc.pl.extract_selected_data(
    adata=clean, min_gene_relative_variance=None, random_seed=RANDOM_SEED,
)
print("selected feature genes:", int(clean.var["selected_gene"].sum()))

leaked = clean.var_names[(clean.var["selected_gene"].values) &
                         (clean.var.get("lateral_gene", pd.Series(False, index=clean.var_names)).values)]
print("Lateral genes that leaked into selection (should be empty):", list(leaked))

# most informative check for a subset: are TCR V/J in top features (can add checks for other genes like ribosomal) 
top_feats = clean.var_names[clean.var["selected_gene"].values]
print("selected genes matching TR[ABGD][VJD]:",
      [g for g in top_feats if any(g.startswith(p) for p in ("TRAV","TRBV","TRGV","TRDV","TRAJ","TRBJ"))])


## 9. Target metacell size & pile size, then run DAC

`TARGET_METACELL_SIZE` / `TARGET_METACELL_UMIS` being passed at top of notebook.

DAC runs for `2 x MIN_PILE` cells. For a small subsets
(pDCs, platelets, plasma cells) it directly goes to `compute_metacells`
call, which is both faster and avoids pile-boundary artefacts on a population that would
otherwise be split across piles for no reason, but for subsets where it is important to tease apart cells,`MIN_PILE` can be dropped to call DAC for splitting for potentially more precise calls


In [ ]:
use_dac = clean.n_obs >= 2 * MIN_PILE
print(f"{clean.n_obs} clean cells -> {'divide_and_conquer' if use_dac else 'direct compute_metacells'}")

if use_dac:
    mc.pl.compute_target_pile_size(
        adata=clean,
        target_metacell_size=TARGET_METACELL_SIZE,
        target_metacell_umis=TARGET_METACELL_UMIS,
        min_target_pile_size=MIN_PILE,
        max_target_pile_size=MAX_PILE,
        target_metacells_in_pile=TARGET_METACELLS_IN_PILE,
    )
    with mc.ut.progress_bar():
        mc.pl.divide_and_conquer_pipeline(
            adata=clean,
            target_metacell_size=TARGET_METACELL_SIZE,
            target_metacell_umis=TARGET_METACELL_UMIS,
            min_target_pile_size=MIN_PILE,
            max_target_pile_size=MAX_PILE,
            target_metacells_in_pile=TARGET_METACELLS_IN_PILE,
            random_seed=RANDOM_SEED,
        )
else:
    with mc.ut.progress_bar():
        mc.pl.compute_metacells(
            adata=clean,
            target_metacell_size=TARGET_METACELL_SIZE,
            target_metacell_umis=TARGET_METACELL_UMIS,
            random_seed=RANDOM_SEED,
        )


## 10. Collect metacells + outlier QC

Metacell names are prefixed here (`T_M...`). Everything downstream keys on `metacell_name`, so
prefixing once at the source means the subset lookup CSV, the subset Seurat object and the global
one can be joined or `rbind`-ed later without silently matching the wrong rows.

Outlier rate is worth watching: it usually rises on a subset because the population is more
homogeneous and the DAC has less signal to separate on. Above ~10% suggests
`TARGET_METACELL_SIZE` is too large for how much structure is left.


In [ ]:
metacells = mc.pl.collect_metacells(clean, name=f"{PROJECT}.subset.metacells", random_seed=RANDOM_SEED)

# prefix so subset metacell IDs can never collide with the global run's T2, M2, ...
metacells.obs_names = [f"{MC_PREFIX}{n}" for n in metacells.obs_names]
if "metacell_name" in metacells.obs:
    metacells.obs["metacell_name"] = metacells.obs_names.astype(str)
_mcn    = clean.obs["metacell_name"].astype(str).to_numpy()
_is_out = clean.obs["metacell"].to_numpy().astype(int) < 0
clean.obs["metacell_name"] = np.where(_is_out, _mcn, np.char.add(MC_PREFIX, _mcn))
assert set(clean.obs["metacell_name"][~_is_out]).issubset(set(metacells.obs_names))

# per-sample outlier stage (metacell < 0), appended onto the subset composition table
grp = clean.obs.groupby("sample_origin", observed=True)["metacell"]
outdf = pd.DataFrame({
    "n_clean_cells": grp.size(),
    "n_outlier": grp.apply(lambda s: int((np.asarray(s) < 0).sum())),
}).reset_index()
outdf["n_in_metacell"] = outdf["n_clean_cells"] - outdf["n_outlier"]
outdf["outlier_pct"] = (100 * outdf["n_outlier"] / outdf["n_clean_cells"]).round(2)

excluded_per_sample = attrition.merge(outdf, on="sample_origin", how="left")
write_xlsx_sheets(QC_XLSX_PATH, {"excluded_per_sample": excluded_per_sample})
print(excluded_per_sample.to_string(index=False))

n_outlier = int((clean.obs["metacell"] < 0).sum())
print(f"\nMetacells: {metacells.n_obs} | outlier cells: {n_outlier} "
      f"({100*n_outlier/clean.n_obs:.2f}%)")
print(f"mean cells per metacell: {(clean.n_obs - n_outlier)/max(metacells.n_obs,1):.1f}")


## 11. Rare gene modules

pulling these lists helps identify rare cell types and also flags potential genes to be moved to lateral lists.

On a subset this often comes back empty, because the rare populations the global run was
detecting (pDC, platelet, plasma) are no longer in the object. An empty result here is
informative rather than a failure.


In [ ]:
#what the rare gene detector actually pulls
rare_var_cols = [c for c in clean.var.columns if "rare" in c.lower()]
rare_obs_cols = [c for c in clean.obs.columns if "rare" in c.lower()]
print("rare-related var columns:", rare_var_cols)
print("rare-related obs columns:", rare_obs_cols)

if "rare_gene" in clean.var:
    print("genes in ANY rare module  (var['rare_gene']):", int(clean.var["rare_gene"].sum()))
if "rare_gene_module" in clean.var:
    v = pd.Series(clean.var["rare_gene_module"]).astype(int)
    print("distinct gene-module indices (var, -1 = none):", sorted(v.unique()))
if "rare_gene_module" in clean.obs:
    c = pd.Series(clean.obs["rare_gene_module"]).astype(int)
    print("cells per rare module (obs, -1 = none):")
    print(c.value_counts().sort_index())
print("rare-related metacells.obs columns:", [x for x in metacells.obs.columns if "rare" in x.lower()])

# per-module cell counts: the per-cell rare index is stored under a different name than the per-gene one
def _find_col(df, names):
    for nm in names:
        if nm in df.columns:
            return nm
    return None

_cell_names = ["rare_gene_module", "cells_rare_gene_module", "rare_cell_module", "cell_rare_gene_module"]
cells_per_module = {}
obs_col = _find_col(clean.obs, _cell_names)
mc_col  = _find_col(metacells.obs, _cell_names)
if obs_col is not None:
    s = pd.Series(clean.obs[obs_col].values).astype(int)
    cells_per_module = {int(m): int((s == m).sum()) for m in s.unique() if m >= 0}
    _src = f"clean.obs['{obs_col}']"
elif mc_col is not None and "grouped" in metacells.obs.columns:
    mm  = pd.Series(metacells.obs[mc_col].values).astype(int)
    grp = pd.Series(metacells.obs["grouped"].values)
    cells_per_module = {int(m): int(grp[mm == m].sum()) for m in mm.unique() if m >= 0}
    _src = f"metacells.obs['{mc_col}'] x grouped"
else:
    _src = "unavailable (n_cells will be NaN)"
print("per-module cell-count source:", _src)

# ---- 2) build the summary from whichever schema is present ----
rows_summary, rows_long = [], []
gmod = pd.Series(clean.var["rare_gene_module"].values, index=clean.var_names).astype(int) \
       if "rare_gene_module" in clean.var else None
per_module_cols = sorted([c for c in clean.var.columns if c.startswith("rare_gene_module_")],
                         key=lambda s: int(s.rsplit("_", 1)[1]))

if gmod is not None and (gmod >= 0).any():                 # integer-index schema (expected)
    for m in sorted(x for x in gmod.unique() if x >= 0):
        genes = gmod.index[gmod == m].tolist()
        rows_summary.append({"module": m, "n_genes": len(genes),
                             "n_cells": cells_per_module.get(int(m), np.nan), "genes": ", ".join(genes)})
        rows_long += [{"module": m, "gene": g} for g in genes]
elif per_module_cols:                                      # per-module boolean-mask schema
    for col in per_module_cols:
        m = int(col.rsplit("_", 1)[1]); genes = clean.var_names[clean.var[col].values].tolist()
        rows_summary.append({"module": m, "n_genes": len(genes),
                             "n_cells": cells_per_module.get(m, np.nan), "genes": ", ".join(genes)})
        rows_long += [{"module": m, "gene": g} for g in genes]

summary_df = pd.DataFrame(rows_summary, columns=["module", "n_genes", "n_cells", "genes"])
long_df    = pd.DataFrame(rows_long, columns=["module", "gene"])
if len(summary_df):
    print(f"\n{len(summary_df)} rare gene modules")
    display(summary_df)
else:
    print("\n>>> No rare gene modules were detected in this run.\n")

write_xlsx_sheets(QC_XLSX_PATH, {"rare_gene_modules": long_df, "rare_gene_summary": summary_df})


In [ ]:
# Check which rare gene modules come from which blocks in case they are contaminants. also to check whether 500 rare gene cap is met
#if there are some repeated problematic blocks
# they can be dropped at top of the file where manual block checking before subsetting happens

def _find_col(df, names):
    for nm in names:
        if nm in df.columns:
            return nm
    return None

_cell_names = ["rare_gene_module", "cells_rare_gene_module", "rare_cell_module", "cell_rare_gene_module"]
obs_col = _find_col(clean.obs, _cell_names)
mc_col  = _find_col(metacells.obs, _cell_names)
print("clean.obs:", [c for c in clean.obs.columns if "rare" in c.lower()])
print("metacells.obs:", [c for c in metacells.obs.columns if "rare" in c.lower()])
print("->", obs_col, "|", mc_col)

if obs_col is not None:
    rm = pd.Series(clean.obs[obs_col].values).astype(int)
    sub = clean.obs.loc[(rm >= 0).to_numpy()].copy()
    sub["rare_module"] = rm[rm >= 0].to_numpy()
    print(sub.groupby(["rare_module", "global_mc_block"], observed=True).size())
    print("\nby global label:")
    print(sub.groupby(["rare_module", "global_broad_label"], observed=True).size())
    rare_cells_by_module = {int(m): list(g.index) for m, g in sub.groupby("rare_module")} # build dict to drop moduyles 
    print("\ncells per rare module:", {m: len(v) for m, v in rare_cells_by_module.items()})
elif mc_col is not None:
    mm = pd.Series(metacells.obs[mc_col].values).astype(int)
    print(metacells.obs.loc[(mm >= 0).to_numpy(), ["grouped"]].assign(rare_module=mm[mm >= 0].to_numpy()))
else:
    print("no per-cell rare module assignment written; use the gene lists only")


#Rare gene has a 500 cap, so check if cap is being met. this can be an issue because if other contaminant genes cause the cap to be met then
# genes teasing apart the actual subset rare cell types can get lost; not true if cap is not met
X = clean.X.tocsc()
n = clean.n_obs
n_expressing = np.diff(X.indptr)
gene_max = np.asarray(X.max(axis=0).todense()).flatten()

candidate = (n_expressing / n < 0.001) & (gene_max >= 7)
print(f"candidate rare genes: {int(candidate.sum())} (cap is 500)")
print(f"cells threshold: {0.001*n:.0f}")


In [ ]:
## OPTIONAL: Drop rare gene module cells if needed, then rerun from "Subset for cell type of interest"
# read the gene lists in summary_df above, then list the module indices that are not T cells
# DROP_MODULES = [0, 1, 2, 3, 4]   # myeloid, mast/basophil, DC

# RARE_CELLS_TO_DROP = [c for m in DROP_MODULES for c in rare_cells_by_module.get(m, [])]
# print(f"{len(RARE_CELLS_TO_DROP)} cells flagged across modules {DROP_MODULES}")
# print(summary_df.loc[summary_df["module"].isin(DROP_MODULES), ["module","n_genes","n_cells","genes"]].to_string(index=False))


## 12. Save both objects

`clean_cells.h5ad` carries the per-cell metacell assignment plus the `lateral_gene` / `noisy_gene`
/ `rare_gene` masks - the 05 Rmd reads those to (a) exclude lateral genes from module HVGs and
(b) pull rare genes. `metacells.h5ad` is the collected metacell object.

`global_broad_label` and `global_metacell_name` ride along so any cell in the subset object can
be traced back to its parent metacell in the global run.


In [ ]:
# named with the subset slug so several lineages can coexist in one tree
clean.write_h5ad(os.path.join(h5ad_dir, f"{PROJECT}.subset.clean_cells.h5ad"))
metacells.write_h5ad(os.path.join(h5ad_dir, f"{PROJECT}.subset.metacells.h5ad"))
print("Saved to", h5ad_dir)
